In [1]:
import pandas as pd

def replace_urls_with_prefixes(text, prefix_dict):
    """Replace URLs with prefixes in a given string.

    Args:
        text (str): String containing SPARQL query or text.
        prefix_dict (dict): Mapping of prefixes to URLs.
    """
    for prefix, url in prefix_dict.items():
        text = text.replace(url, prefix)
    return text


def replace_prefixes_with_urls(text, prefix_dict):
    """Replace prefixes with URLs in a given string.

    Args:
        text (str): String containing SPARQL query or text.
        prefix_dict (dict): Mapping of prefixes to URLs.
    """
    for prefix, url in prefix_dict.items():
        text = text.replace(prefix, url)
    return text


def parse_sparql_output(query_output: dict):
    """Parse query JSON output to dataframe.

    Args:
        query_output (dict): The output from a SPARQL query in JSON format

    Returns:
        pd.DataFrame: the parsed results as a pandas DataFrame
    """
    columns = query_output['head']['vars']
    rows = []
    for result in query_output['results']['bindings']:
        row = {}
        for col in columns:
            row[col] = result[col]['value'] if col in result else None
        rows.append(row)
    df = pd.DataFrame(rows, columns=columns)
    return df

In [2]:
import pandas as pd

def replace_urls_with_prefixes(text, prefix_dict):
    """Replace URLs with prefixes in a given string.

    Args:
        text (str): String containing SPARQL query or text.
        prefix_dict (dict): Mapping of prefixes to URLs.
    """
    for prefix, url in prefix_dict.items():
        text = text.replace(url, prefix)
    return text


def replace_prefixes_with_urls(text, prefix_dict):
    """Replace prefixes with URLs in a given string.

    Args:
        text (str): String containing SPARQL query or text.
        prefix_dict (dict): Mapping of prefixes to URLs.
    """
    for prefix, url in prefix_dict.items():
        text = text.replace(prefix, url)
    return text


def parse_sparql_output(query_output: dict):
    """Parse query JSON output to dataframe.

    Args:
        query_output (dict): The output from a SPARQL query in JSON format

    Returns:
        pd.DataFrame: the parsed results as a pandas DataFrame
    """
    columns = query_output['head']['vars']
    rows = []
    for result in query_output['results']['bindings']:
        row = {}
        for col in columns:
            row[col] = result[col]['value'] if col in result else None
        rows.append(row)
    df = pd.DataFrame(rows, columns=columns)
    return df

In [3]:
# SPARQL

from SPARQLWrapper import SPARQLWrapper, JSON

def execute_sparql(url: str, query: str):
    """    
    Executes a SPARQL query against a specified endpoint.

    This function sends a SPARQL query to the given endpoint URL and returns the results in JSON format.

    Args:
        url (str): The URL of the SPARQL query endpoint.
        query (str): The SPARQL query to be executed.

    Returns:
        dict: The results of the SPARQL query in JSON format.
    """
    wrapper = SPARQLWrapper(url)
    wrapper.setQuery(query)
    wrapper.setReturnFormat(JSON)
    results = wrapper.query().convert()

    return results


def execute_parse_sparql(url, queries):
    """Helper function for retrieve_context().

    Args:
        url (_type_): _description_
        queries (_type_): _description_

    Returns:
        _type_: _description_
    """
    results = []
    for index, query in enumerate(queries):
        result = execute_sparql(url=url, query=PREFIXES+query)
        table = parse_sparql_output(result).to_string()
        results.append(f"Triples: {index}\n{table}")
    subgraph = "\n".join(results)
    subgraph = replace_urls_with_prefixes(subgraph, prefix_dict)
    return subgraph


PREFIXES = """
  PREFIX : <http://www.co-ode.org/ontologies/pizza#> 
  PREFIX dc: <http://purl.org/dc/elements/1.1/> 
  PREFIX owl: <http://www.w3.org/2002/07/owl#> 
  PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> 
  PREFIX xml: <http://www.w3.org/XML/1998/namespace> 
  PREFIX xsd: <http://www.w3.org/2001/XMLSchema#> 
  PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 
  PREFIX skos: <http://www.w3.org/2004/02/skos/core#> 
  PREFIX pizza: <http://www.co-ode.org/ontologies/pizza/pizza.owl#> 
  PREFIX terms: <http://purl.org/dc/terms/> 
  BASE <http://www.co-ode.org/ontologies/pizza#> 
"""

prefix_dict = {
    ":": "http://www.co-ode.org/ontologies/pizza#",
    "dc:": "http://purl.org/dc/elements/1.1/",
    "owl:": "http://www.w3.org/2002/07/owl#",
    "rdf:": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "xml:": "http://www.w3.org/XML/1998/namespace",
    "xsd:": "http://www.w3.org/2001/XMLSchema#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "skos:": "http://www.w3.org/2004/02/skos/core#",
    "pizza:": "http://www.co-ode.org/ontologies/pizza/pizza.owl#",
    "terms:": "http://purl.org/dc/terms/",
    "base:": "http://www.co-ode.org/ontologies/pizza#"
}

EXTRACTION_QUERY = """
SELECT DISTINCT ?individual
WHERE {{
{{
 SELECT ?individual
 WHERE {{
   ?individual rdfs:subClassOf* {pizza_class} .
 }}
}}
UNION
{{
 SELECT DISTINCT ?individual
 WHERE {{
   ?individual rdf:type owl:Class ;
           owl:equivalentClass ?equivClass .

   ?equivClass owl:intersectionOf ?list .
   ?list rdf:rest*/rdf:first {pizza_class} .
 }}
}}
}}
"""

In [5]:
# UTILS
from strictjson import strict_json
from openai import OpenAI
import chromadb

ER_SYSTEM_PROMPT = """
You are an expert in the gastronomy domain with extensive knowledge of various pizzas and toppings. Your task is to accurately identify and extract the names of machines and parts from given text. You should 
focus on recognizing specific terminology and contextually relevant phrases that pertain to pizza names and their toppings 
within the gastronomy sector. Your output should be in JSON format, categorizing the identified terms into "pizza" 
and "topping". The output should be strictly the JSON object without any additional commentary or explanation.
"""

ER_USER_PROMPT = """
I want you to identify the pizza and topping of the following input.

Input: {input}
Output: 
"""

def strictjson_llm(system_prompt: str, user_prompt: str):
    """LLM function call to be passed to strictjson.strict_json().
    Refer to https://github.com/tanchongmin/strictjson/blob/main/strictjson/base.py#L319

    Args:
        system_prompt (str): System prompt.
        user_prompt (str): User prompt.

    Returns:
        str: LLM response string.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]   

    client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

    response = client.chat.completions.create(
        model="lmstudio-community/Meta-Llama-3-8B-Instruct-GGUF",
        messages=messages,
        temperature=0.7,
    )
    
    return response.choices[0].message.content

def entity_recognition(input: str) -> dict:
    """
    Performs entity recognition on the provided input string using a large language model (LLM).

    This function identifies and categorizes entities within the user input, specifically separating them into machine and part.

    Multiple 

    Args:
        input (str): The input string to be analyzed by the LLM.

    Returns:
        dict: A dictionary with two keys, 'machines' and 'parts', containing the identified machine names and part names respectively.

    Example:
        Input: "Can Creality Ender manufacture flange?"
        Output: {'machines': ['Creality Ender'], 'parts': ['flange']}

    """
    response = strict_json(
        system_prompt=ER_SYSTEM_PROMPT,
        user_prompt=ER_USER_PROMPT.format(input=input),
        output_format={"pizza": "Array of pizza", 
                       "topping": "Array of topping"},
        llm=strictjson_llm
    )

    return response

### Context retrieval

In [7]:
entities = entity_recognition("Can an American pizza use a cheese topping?")
path = '../../manon_chat_interface/data/vectorstores/entities'

In [8]:
client = chromadb.PersistentClient(path=path)
if entities["pizza"]:
    collection = client.get_or_create_collection("pizza_collection")

In [10]:
embeddings = collection.query(
    query_texts=entities["pizza"],
    n_results=1
)
embeddings

{'ids': [['0c3340e5-7cbf-58e5-8ca9-64b4b1893367']],
 'distances': [[0.0]],
 'metadatas': [[{'IRI': 'http://www.co-ode.org/ontologies/pizza/pizza.owl#American'}]],
 'embeddings': None,
 'documents': [['American']],
 'uris': None,
 'data': None}

### SPARQL

In [21]:
import pandas as pd

def parse_sparql_output(query_output: dict):
    """_summary_

    Args:
        query_output (dict): The output from a SPARQL query in JSON format

    Returns:
        pd.DataFrame: the parsed results as a pandas DataFrame
    """
    columns = query_output['head']['vars']
    rows = []
    for result in query_output['results']['bindings']:
        row = {}
        for col in columns:
            row[col] = result[col]['value'] if col in result else None
        rows.append(row)
    df = pd.DataFrame(rows, columns=columns)
    return df

In [19]:
from SPARQLWrapper import SPARQLWrapper, JSON

import re

# Define the SPARQL endpoint
sparql = SPARQLWrapper("http://localhost:3030/pizza/query")

# Set the query
sparql.setQuery(query)

# Set the return format to JSON
sparql.setReturnFormat(JSON)

# Execute the query and get the results
results = sparql.query().convert()

# bindings = results['results']['bindings']
# pizza_IRIs = [binding['individual']['value'] for binding in bindings]
# pizza_names = [re.split('#', iri)[-1] for iri in pizza_IRIs]

In [7]:
URL = "http://localhost:3030/pizza/query"
iri = "pizza:Pizza"

results = execute_sparql(url=URL, query=PREFIXES+EXTRACTION_QUERY.format(pizza_class=iri))

In [20]:
results

{'head': {'vars': ['class', 'label']},
 'results': {'bindings': [{'class': {'type': 'uri',
     'value': 'http://www.co-ode.org/ontologies/pizza/pizza.owl#Pizza'},
    'label': {'type': 'literal', 'xml:lang': 'en', 'value': 'Pizza'}},
   {'class': {'type': 'uri',
     'value': 'http://www.co-ode.org/ontologies/pizza/pizza.owl#PizzaBase'},
    'label': {'type': 'literal', 'xml:lang': 'pt', 'value': 'BaseDaPizza'}},
   {'class': {'type': 'uri',
     'value': 'http://www.co-ode.org/ontologies/pizza/pizza.owl#PizzaBase'},
    'label': {'type': 'literal', 'xml:lang': 'en', 'value': 'PizzaBase'}},
   {'class': {'type': 'uri',
     'value': 'http://www.co-ode.org/ontologies/pizza/pizza.owl#Food'},
    'label': {'type': 'literal', 'xml:lang': 'en', 'value': 'Food'}},
   {'class': {'type': 'uri',
     'value': 'http://www.co-ode.org/ontologies/pizza/pizza.owl#Spiciness'},
    'label': {'type': 'literal', 'xml:lang': 'en', 'value': 'Spiciness'}},
   {'class': {'type': 'uri',
     'value': 'http:

In [25]:
df = parse_sparql_output(results)

In [34]:
prefix_dict = {
    ":": "http://www.co-ode.org/ontologies/pizza#",
    "dc:": "http://purl.org/dc/elements/1.1/",
    "owl:": "http://www.w3.org/2002/07/owl#",
    "rdf:": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "xml:": "http://www.w3.org/XML/1998/namespace",
    "xsd:": "http://www.w3.org/2001/XMLSchema#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "skos:": "http://www.w3.org/2004/02/skos/core#",
    "pizza:": "http://www.co-ode.org/ontologies/pizza/pizza.owl#",
    "terms:": "http://purl.org/dc/terms/",
    "base:": "http://www.co-ode.org/ontologies/pizza#"
}

def replace_urls_with_prefixes(text, prefix_dict):
    """Replace URLs with prefixes in a given string.

    Args:
        query (str): String containing SPARQL query or text.
        prefix_dict (dict): Mapping of prefixes to URLs.
    """
    for prefix, url in prefix_dict.items():
        query = query.replace(url, prefix)
    return query


def replace_prefixes_with_urls(query, prefix_dict):
    """Replace prefixes with URLs in a given string.

    Args:
        query (str): String containing SPARQL query or text.
        prefix_dict (dict): Mapping of prefixes to URLs.
    """
    for prefix, url in prefix_dict.items():
        query = query.replace(prefix, url)
    return query

### Generate n-hop

In [1]:
PREFIXES = """
PREFIX : <http://www.co-ode.org/ontologies/pizza#> 
PREFIX dc: <http://purl.org/dc/elements/1.1/> 
PREFIX owl: <http://www.w3.org/2002/07/owl#> 
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> 
PREFIX xml: <http://www.w3.org/XML/1998/namespace> 
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#> 
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 
PREFIX skos: <http://www.w3.org/2004/02/skos/core#> 
PREFIX pizza: <http://www.co-ode.org/ontologies/pizza/pizza.owl#> 
PREFIX terms: <http://purl.org/dc/terms/> 
BASE <http://www.co-ode.org/ontologies/pizza#> 
"""



In [7]:
ONE_HOP = """
SELECT ?subject ?predicate ?object WHERE {{
  {{
    ?s ?p {IRI} .
    BIND(?s AS ?subject)
    BIND(?p AS ?predicate)
    BIND({IRI} AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    {IRI} ?p ?o .
    BIND({IRI} AS ?subject)
    BIND(?p AS ?predicate)
    BIND(?o AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
}}
"""


In [9]:
TWO_HOP = """
SELECT ?subject ?predicate ?object WHERE {{
  {{
    ?s ?p {IRI} .
    BIND(?s AS ?subject)
    BIND(?p AS ?predicate)
    BIND({IRI} AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    ?s ?p {IRI} .
    ?s1 ?p1 ?s .
    BIND(?s1 AS ?subject)
    BIND(?p1 AS ?predicate)
    BIND(?s AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    {IRI} ?p ?o .
    BIND({IRI} AS ?subject)
    BIND(?p AS ?predicate)
    BIND(?o AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    {IRI} ?p ?o .
    ?o ?p1 ?o1 .
    BIND(?o AS ?subject)
    BIND(?p1 AS ?predicate)
    BIND(?o1 AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
}}
"""

In [1]:
THREE_HOP = """
SELECT ?subject ?predicate ?object WHERE {{
  {{
    ?s ?p {IRI} .
    BIND(?s AS ?subject)
    BIND(?p AS ?predicate)
    BIND({IRI} AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    ?s ?p {IRI} .
    ?s1 ?p1 ?s .
    BIND(?s1 AS ?subject)
    BIND(?p1 AS ?predicate)
    BIND(?s AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    ?s ?p {IRI} .
    ?s1 ?p1 ?s .
    ?s2 ?p2 ?s1 .
    BIND(?s2 AS ?subject)
    BIND(?p2 AS ?predicate)
    BIND(?s1 AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    {IRI} ?p ?o .
    BIND({IRI} AS ?subject)
    BIND(?p AS ?predicate)
    BIND(?o AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    {IRI} ?p ?o .
    ?o ?p1 ?o1 .
    BIND(?o AS ?subject)
    BIND(?p1 AS ?predicate)
    BIND(?o1 AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    {IRI} ?p ?o .
    ?o ?p1 ?o1 .
    ?o1 ?p2 ?o2 .
    BIND(?o1 AS ?subject)
    BIND(?p2 AS ?predicate)
    BIND(?o2 AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
}}
"""


### Test generated SPARQL executability

In [9]:
result = {'sparql_query': "\nPREFIX pizza: <http://www.co-ode.org/ontologies/pizza#>\nPREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n\nSELECT ?p\nWHERE {\n  ?p a pizza:Pizza.\n  ?p a pizza:AmercianPizza.\n  OPTIONAL { ?p pizza:hasTopping ?t. }\n  FILTER ( regex(?t, 'Mozzarella', 'i') )\n}\n", 'explanation': 'This SPARQL query first selects pizzas that are of type American Pizza and then optionally includes the toppings of these pizzas. It filters for Mozzarella topping in a case-insensitive manner.'}

sparql_query = result['sparql_query']
explanation = result['explanation']

In [10]:
print(sparql_query)


PREFIX pizza: <http://www.co-ode.org/ontologies/pizza#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?p
WHERE {
  ?p a pizza:Pizza.
  ?p a pizza:AmercianPizza.
  OPTIONAL { ?p pizza:hasTopping ?t. }
  FILTER ( regex(?t, 'Mozzarella', 'i') )
}



In [6]:
result = execute_parse_sparql(url="http://localhost:3030/pizza/query", queries=[sparql_query])

In [7]:
result

'Triples: 0\nEmpty DataFrame\nColumns: [p, o]\nIndex: []'